In [ ]:
# !pip install openai
# !pip install python-dotenv

In [1]:
import os
import openai
import sys
sys.path.append('./')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.environ['OPENAI_API_KEY']

# Method 1 - openai library 이용
## role of the messages author

OpenAI API에서 각 메시지의 저자(`author`) 역할은 다음과 같은 의미를 가집니다:

1. **system**: 시스템 관련 메시지나 지침. 예를 들면, 어시스턴트에게 특정 작업을 수행하는 방법에 대한 지침을 제공하는 경우에 사용됩니다. 이러한 지침은 어시스턴트의 응답을 안내하거나 제한할 수 있습니다.

2. **user**: 사용자가 입력한 메시지. 대체로 사용자의 질문, 요청, 지침 등을 나타냅니다.

3. **assistant**: 어시스턴트(즉, 모델)가 반환하는 응답 또는 메시지. 사용자의 질문에 대한 답변, 제안, 설명 등을 포함할 수 있습니다.

4. **function**: API의 이전 버전에서 사용됐던 것으로, 특정 기능이나 동작을 어시스턴트에게 요청할 때 사용되었습니다. 최신 버전에서는 일반적으로 사용되지 않습니다.

이러한 역할은 대화형 세션에서 메시지의 발신자와 의도를 구별하는 데 도움을 줍니다. API는 이 정보를 사용하여 적절한 방식으로 반응하거나 응답합니다.

In [2]:
completion = openai.ChatCompletion.create(
  model="gpt-3.5-turbo",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"}
  ]
)

print(completion.choices[0].message)

APIRemovedInV1: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742


In [4]:
# assistant 에 대한 지침이 되는 context 정보를 제공
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
]

while True:
    # 사용자로부터 메시지 입력 받기
    message = input("User : ")
    
    # 사용자가 'stop'이라고 입력하면 대화 종료
    if message.lower() == "stop":
        break
    else:
        # 사용자의 메시지를 메시지 목록에 추가
        messages.append(
             {"role": "user", "content": message}
        )
        
        # OpenAI API를 사용하여 대화형 완성 진행
        completion = openai.ChatCompletion.create(
          model="gpt-3.5-turbo",
          messages=messages
        )
        
    # 모델의 답변 추출
    reply = completion.choices[0].message.content
    
    # 모델의 답변 출력
    print(f"ChatGPT: {reply}")
    
    # 모델의 답변을 메시지 목록에 추가
    messages.append({"role": "assistant", "content": reply})

User :  stop


In [4]:
messages

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': '안녕. 너는 누구야?'},
 {'role': 'assistant', 'content': '안녕하세요! 저는 도움이 되는 어시스턴트입니다. 무엇을 도와드릴까요?'},
 {'role': 'user', 'content': '서울에서 칼국수 제일 잘하는 집이 어디야?'},
 {'role': 'assistant',
  'content': '서울에는 많은 칼국수 맛집이 있지만, 그 중에서도 유명한 몇 곳을 소개해드릴게요.\n\n1. 국민옥: 서울의 대표적인 칼국수 맛집으로 유명하며, 옛날 방식 그대로 손으로 만든 면과 깊은 육수가 특징입니다. 서울 종로구 북촌로5길 15에 위치해 있습니다.\n\n2. 남원추어칼국수: 전통적인 남원식 추어칼국수를 맛볼 수 있는 곳으로 유명합니다. 구로구 구로동로4길 37에 위치해 있습니다.\n\n3. 대전왕갈비칼국수: 대전에서 유명한 갈비와 칼국수를 한 곳에서 즐길 수 있는 집입니다. 용산구 서빙고로 17길 15에 위치해 있습니다.\n\n이 외에도 다양한 칼국수 맛집이 있으니, 개인적인 취향에 맞게 선택하시면 됩니다. 맛있는 식사 되세요!'}]

# Method 2 - API Endpoint 이용

payload"는 주로 HTTP 요청을 보낼 때 전달되는 데이터를 의미합니다. API 호출에서 "payload"는 주로 POST 요청의 본문에 포함된 데이터를 나타냅니다.  

- temperature : 사용할 샘플링 온도는 0에서 2 사이입니다. 0.8과 같이 값이 높을수록 출력이 더 무작위로 생성(Hallucination 발생 가능)되고, 0.2와 같이 값이 낮을수록 더 집중적이고 결정적이게 됩니다. 

- top_p : temperature의 대안으로, 확률 top_p인 토큰의 결과를 고려합니다. 따라서 0.1은 상위 10% 확률을 구성하는 토큰만 고려된다는 의미입니다.

**temperature 와 top_p 중 하나만 변경하고, 둘 다 변경하지는 않는 것이 좋습니다.**  

- n:  각 입력 메시지에 대해 생성할 chat completion 선택 수입니다.  

-  presence_penalty : -2.0과 2.0 사이의 숫자입니다. 양수 값은 지금까지 텍스트에 나타나는지 여부에 따라 새 토큰에 불이익을 주어 모델이 새로운 주제에 관해 이야기할 가능성을 높입니다.

- frequency_penalty : -2.0과 2.0 사이의 숫자입니다. 양수 값은 지금까지 텍스트의 기존 빈도를 기반으로 새 토큰에 불이익을 주어 모델이 동일한 줄을 그대로 반복할 가능성을 줄입니다.  

Bearer는 HTTP 인증 스키마 중 하나입니다. 웹 서비스에서 사용하는 토큰 기반의 인증 방식 중 가장 일반적인 방식입니다. Bearer를 사용하면, 클라이언트는 해당 토큰을 포함하여 서버에 요청을 보낼 수 있으며, 서버는 이 토큰을 검증하여 해당 클라이언트의 요청을 인증합니다. Bearer {토큰} 형식에서 {토큰} 부분은 실제 API를 사용하기 위한 인증 토큰을 나타냅니다.

In [5]:
import requests  

# OpenAI API의 URL을 정의
URL = "https://api.openai.com/v1/chat/completions"

# 요청에 필요한 데이터를 payload 변수에 저장
payload = {
    "model": "gpt-3.5-turbo",  
    "messages": [{"role": "user", "content": f"Who is the most successful chess champion in the world ?"}],  
    "temperature": 1.0,  
    "top_p": 1.0,  
    "n": 1,  
    "stream": False,     # 스트림모드 사용 여부
    "presence_penalty": 0,  # 결과의 일관성에 영향
    "frequency_penalty": 0,  # 단어의 사용 빈도에 영향
}

# 요청 헤더에 내용 유형 및 인증 키 설정
headers = {
    "Content-Type": "application/json",  # 요청 본문의 유형을 JSON으로 지정
    "Authorization": f"Bearer {openai.api_key}"  # 인증 키 포함
}

# requests.post를 사용하여 OpenAI API에 POST 요청을 보냅니다.
response = requests.post(URL, headers=headers, json=payload, stream=False)

In [6]:
response

<Response [200]>

### response 내용
1. **choices**: API 응답의 주요 내용이 포함된 배열입니다.
   - **finish_reason**: 응답이 완료된 이유입니다. 'stop'은 출력이 종료된 이유가 됩니다.
   - **index**: 선택 사항의 인덱스 번호입니다.
   - **message**: 사용자 또는 보조프로그램의 메시지 내용입니다.
     - **content**: 보조프로그램이 생성한 메시지 내용입니다. 
     - **role**: 이 메시지를 생성한 주체입니다.  'assistant'는 메시지가 OpenAI의 assistena program으로부터 생성되었다는 것을 의미합니다.

2. **created**: 응답이 생성된 시간의 타임스탬프입니다.

3. **id**: 응답에 할당된 고유 식별자입니다.

4. **model**: 사용된 모델의 이름입니다.

5. **object**: 이 객체의 유형을 나타냅니다. 'chat.completion'은 채팅 완료 응답임을 나타냅니다.

6. **usage**: 이 요청에서 사용된 토큰의 수를 나타내는 정보입니다.
   - **completion_tokens**: 응답 내용에서 사용된 토큰 수입니다.
   - **prompt_tokens**: 사용자의 원래 질문에서 사용된 토큰 수입니다.
   - **total_tokens**: 전체 토큰 수입니다.

In [7]:
import pprint
import json

# 응답 내용을 JSON으로 파싱
response_json = response.json()

# 파싱된 JSON 내용을 pretty-print로 출력
pprint.pprint(response_json)

{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'message': {'content': 'The most successful chess champion in '
                                     'the world, based on the number of '
                                     'official World Chess Championship titles '
                                     'won, is Garry Kasparov. He became the '
                                     'youngest World Chess Champion in history '
                                     'in 1985 and went on to win the '
                                     'championship title several times.',
                          'role': 'assistant'}}],
 'created': 1694504455,
 'id': 'chatcmpl-7xsIxXPgm4eXbt1KVzxCsca90swTU',
 'model': 'gpt-3.5-turbo-0613',
 'object': 'chat.completion',
 'usage': {'completion_tokens': 52, 'prompt_tokens': 18, 'total_tokens': 70}}


In [19]:
print(response_json['choices'][0]['message']['content'])

The most successful chess champion in the world, based on the number of official World Chess Championship titles won, is Garry Kasparov. He became the youngest World Chess Champion in history in 1985 and went on to win the championship title several times.
